=============================================================
BOOKLY - Model Training & Comparison
=============================================================
Input  : X_train.csv, X_test.csv, y_train.csv, y_test.csv
Output : bookly_model.pkl
         plot_06_model_comparison.png
         plot_07_tuning_results.png
         plot_08_actual_vs_predicted.png
         plot_09_residuals.png
         plot_10_learning_curve.png

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LinearRegression, Ridge
from sklearn.tree             import DecisionTreeRegressor
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors        import KNeighborsRegressor
from sklearn.model_selection  import cross_val_score, GridSearchCV, learning_curve
from sklearn.metrics          import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
BASELINE_RMSE = 0.2996

In [3]:
# Loading
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}\n")


def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'RMSE':  round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        'MAE':   round(float(mean_absolute_error(y_true, y_pred)), 4),
        'R²':    round(float(r2_score(y_true, y_pred)), 4),
    }

X_train: (8831, 9)  |  X_test: (2208, 9)
Features: ['num_pages', 'pub_year', 'series_num', 'title_words', 'num_authors', 'log_ratings_count', 'log_text_reviews_count', 'publisher_enc', 'author_enc']



In [4]:
# Models definition
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LinearRegression()),
    ]),
    'Ridge (α=1)': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0)),
        # L2 regularisation shrinks coefficients; good when
        # publisher_enc and author_enc are correlated.
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  DecisionTreeRegressor(max_depth=8, random_state=42)),
        # max_depth=8 prevents pure training-set memorisation.
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  RandomForestRegressor(n_estimators=100, random_state=42)),
        # Averaging 100 trees reduces variance vs a single tree.
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05,
            max_depth=4, random_state=42)),
        # Builds trees sequentially, each correcting previous errors.
    ]),
    'KNN (k=7)': Pipeline([
        ('scaler', StandardScaler()),
        ('model',  KNeighborsRegressor(n_neighbors=7)),
        # MUST be scaled: without it, num_pages (0-6576) dwarfs
        # author_enc (3.6-4.4) in distance calculations.
    ]),
}

In [5]:
# Training & Evaluation
print(f"{'':2}{'Model':23s}  {'RMSE':>8} {'MAE':>8} {'R²':>8}  {'Time':>6}")
print("─" * 60)

results     = []
fitted_pipes = {}

for name, pipe in models.items():
    t0   = time.time()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    m    = evaluate(name, y_test, pred)
    m['Time'] = round(time.time() - t0, 1)
    results.append(m)
    fitted_pipes[name] = pipe

    beat = '✓' if m['RMSE'] < BASELINE_RMSE else '✗'
    print(f"{beat} {name:23s}  {m['RMSE']:.4f}   {m['MAE']:.4f}   {m['R²']:.4f}  {m['Time']}s")

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
print(f"\nBaseline (predict mean always): RMSE = {BASELINE_RMSE}")
print(f"\nRanked by RMSE:")
print(results_df[['Model','RMSE','MAE','R²']].to_string(index=False))

  Model                        RMSE      MAE       R²    Time
────────────────────────────────────────────────────────────
✓ Linear Regression        0.2588   0.1888   0.2509  0.3s
✓ Ridge (α=1)              0.2588   0.1888   0.2509  0.0s
✓ Decision Tree            0.2674   0.1905   0.2005  0.1s
✓ Random Forest            0.2576   0.1844   0.2580  4.1s
✓ Gradient Boosting        0.2576   0.1837   0.2580  3.6s
✓ KNN (k=7)                0.2632   0.1891   0.2252  0.2s

Baseline (predict mean always): RMSE = 0.2996

Ranked by RMSE:
            Model   RMSE    MAE     R²
    Random Forest 0.2576 0.1844 0.2580
Gradient Boosting 0.2576 0.1837 0.2580
      Ridge (α=1) 0.2588 0.1888 0.2509
Linear Regression 0.2588 0.1888 0.2509
        KNN (k=7) 0.2632 0.1891 0.2252
    Decision Tree 0.2674 0.1905 0.2005


In [6]:
# Cross-validation
print("\n── 5-Fold Cross-Validation (on training set) ──")
print(f"{'Model':25s}  {'CV RMSE':>10}  {'± std':>8}")
print("─" * 50)

cv_scores_all = {}
for name, pipe in fitted_pipes.items():
    cv = -cross_val_score(pipe, X_train, y_train,
                          cv=5, scoring='neg_root_mean_squared_error',
                          n_jobs=-1)
    cv_scores_all[name] = cv
    print(f"  {name:23s}  {cv.mean():.4f}    ±{cv.std():.4f}")


── 5-Fold Cross-Validation (on training set) ──
Model                         CV RMSE     ± std
──────────────────────────────────────────────────
  Linear Regression        0.2043    ±0.0067
  Ridge (α=1)              0.2043    ±0.0067
  Decision Tree            0.1843    ±0.0064
  Random Forest            0.1666    ±0.0057
  Gradient Boosting        0.1670    ±0.0045
  KNN (k=7)                0.1904    ±0.0073


In [7]:
# Hyperparameter tuning
print("\n── GridSearchCV Tuning (Random Forest & Gradient Boosting) ──\n")

# Random Forest grid
rf_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [10, 20, None],
}
gs_rf = GridSearchCV(
    fitted_pipes['Random Forest'], rf_grid,
    cv=3, scoring='neg_root_mean_squared_error', n_jobs=2)
gs_rf.fit(X_train, y_train)
rf_tuned_pred = gs_rf.best_estimator_.predict(X_test)
rf_tuned_m    = evaluate('RF (tuned)', y_test, rf_tuned_pred)
print(f"Random Forest best params : {gs_rf.best_params_}")
print(f"  CV RMSE : {-gs_rf.best_score_:.4f}")
print(f"  Test    : RMSE={rf_tuned_m['RMSE']} MAE={rf_tuned_m['MAE']} R²={rf_tuned_m['R²']}\n")

# Gradient Boosting grid
gb_grid = {
    'model__n_estimators':  [100, 200],
    'model__learning_rate': [0.05, 0.10],
    'model__max_depth':     [3, 5],
}
gs_gb = GridSearchCV(
    fitted_pipes['Gradient Boosting'], gb_grid,
    cv=3, scoring='neg_root_mean_squared_error', n_jobs=2)
gs_gb.fit(X_train, y_train)
gb_tuned_pred = gs_gb.best_estimator_.predict(X_test)
gb_tuned_m    = evaluate('GB (tuned)', y_test, gb_tuned_pred)
print(f"Gradient Boosting best params : {gs_gb.best_params_}")
print(f"  CV RMSE : {-gs_gb.best_score_:.4f}")
print(f"  Test    : RMSE={gb_tuned_m['RMSE']} MAE={gb_tuned_m['MAE']} R²={gb_tuned_m['R²']}")



── GridSearchCV Tuning (Random Forest & Gradient Boosting) ──

Random Forest best params : {'model__max_depth': 20, 'model__n_estimators': 200}
  CV RMSE : 0.1671
  Test    : RMSE=0.2573 MAE=0.184 R²=0.2596

Gradient Boosting best params : {'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 200}
  CV RMSE : 0.1672
  Test    : RMSE=0.2573 MAE=0.1835 R²=0.2598


In [8]:
if rf_tuned_m['RMSE'] <= gb_tuned_m['RMSE']:
    best_name, best_m, best_pred = 'Random Forest (tuned)', rf_tuned_m, rf_tuned_pred
    best_model = gs_rf.best_estimator_
else:
    best_name, best_m, best_pred = 'Gradient Boosting (tuned)', gb_tuned_m, gb_tuned_pred
    best_model = gs_gb.best_estimator_

improvement = (BASELINE_RMSE - best_m['RMSE']) / BASELINE_RMSE * 100
residuals   = y_test.values - best_pred

print(f"""
╔══════════════════════════════════════════════════════╗
║  WINNER: {best_name:<44}║
║  RMSE        : {best_m['RMSE']:<38}║
║  MAE         : {best_m['MAE']:<38}║
║  R²          : {best_m['R²']:<38}║
║  vs baseline : {improvement:.1f}% improvement over RMSE={BASELINE_RMSE}{'':14}║
╚══════════════════════════════════════════════════════╝
""")
print(f"  In plain English: predictions are off by ±{best_m['MAE']:.2f} stars on average.")
print(f"  The model explains {best_m['R²']*100:.1f}% of rating variance.")
print(f"  The remaining {(1-best_m['R²'])*100:.1f}% is taste, social trends, review text not in the data.")


╔══════════════════════════════════════════════════════╗
║  WINNER: Random Forest (tuned)                       ║
║  RMSE        : 0.2573                                ║
║  MAE         : 0.184                                 ║
║  R²          : 0.2596                                ║
║  vs baseline : 14.1% improvement over RMSE=0.2996              ║
╚══════════════════════════════════════════════════════╝

  In plain English: predictions are off by ±0.18 stars on average.
  The model explains 26.0% of rating variance.
  The remaining 74.0% is taste, social trends, review text not in the data.


In [ ]:
# Plotting
# ── Plot 06: All models comparison ──────────────────────────
pre_tune_df = results_df.copy()
colors = sns.color_palette('RdYlGn', len(pre_tune_df))[::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Comparison  6 Algorithms (pre-tuning)', fontsize=13, fontweight='bold')

for ax, metric, lower in zip(axes, ['RMSE','MAE','R²'], [True, True, False]):
    vals   = pre_tune_df[metric].values
    models_ = pre_tune_df['Model'].values
    bars   = ax.barh(models_, vals, color=colors, edgecolor='white')
    if metric == 'RMSE':
        ax.axvline(BASELINE_RMSE, color='crimson', linestyle='--', lw=1.5,
                   label=f'Naive baseline ({BASELINE_RMSE})')
        ax.legend(fontsize=8)
    ax.set_title(f'{metric}  ({"lower" if lower else "higher"} is better)', fontweight='bold')
    ax.set_xlabel(metric)
    for bar, val in zip(bars, vals):
        offset = 0.001 if metric != 'R²' else 0.002
        ax.text(val + offset, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('plot_06_model_comparison.png')
plt.close()
print("\n[Saved] plot_06_model_comparison.png")

# ── Plot 07: Tuning effect ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(2)
w = 0.35
before = [results_df.loc[results_df['Model']=='Random Forest','RMSE'].values[0],
          results_df.loc[results_df['Model']=='Gradient Boosting','RMSE'].values[0]]
after  = [rf_tuned_m['RMSE'], gb_tuned_m['RMSE']]
b1 = ax.bar(x - w/2, before, w, label='Default params', color='steelblue', edgecolor='white')
b2 = ax.bar(x + w/2, after,  w, label='After GridSearchCV', color='teal', edgecolor='white')
ax.axhline(BASELINE_RMSE, color='crimson', linestyle='--', lw=1.2,
           label=f'Naive baseline ({BASELINE_RMSE})')
ax.set_xticks(x); ax.set_xticklabels(['Random Forest', 'Gradient Boosting'])
ax.set_ylabel('Test RMSE'); ax.set_ylim(0.245, 0.31)
ax.set_title('Effect of Hyperparameter Tuning', fontweight='bold')
ax.legend()
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{bar.get_height():.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_07_tuning_results.png')
plt.close()
print("[Saved] plot_07_tuning_results.png")

# ── Plot 08: Actual vs Predicted ─────────────────────────────
lims = [min(y_test.min(), best_pred.min())-0.1,
        max(y_test.max(), best_pred.max())+0.1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Actual vs Predicted  {best_name}', fontsize=13, fontweight='bold')

axes[0].scatter(y_test, best_pred, alpha=0.2, s=10, color='steelblue')
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Rating'); axes[0].set_ylabel('Predicted Rating')
axes[0].set_title('Scatter  (points on diagonal = perfect)'); axes[0].legend()
axes[0].annotate(f'RMSE = {best_m["RMSE"]}\nR²   = {best_m["R²"]}',
    xy=(0.05, 0.88), xycoords='axes fraction', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

hb = axes[1].hexbin(y_test, best_pred, gridsize=30, cmap='Blues', mincnt=1)
axes[1].plot(lims, lims, 'r--', lw=1.5)
axes[1].set_xlabel('Actual Rating'); axes[1].set_ylabel('Predicted Rating')
axes[1].set_title('Density (hexbin)    where predictions cluster')
fig.colorbar(hb, ax=axes[1], label='Count')
plt.tight_layout()
plt.savefig('plot_08_actual_vs_predicted.png')
plt.close()
print("[Saved] plot_08_actual_vs_predicted.png")

# ── Plot 09: Residuals ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'Residual Analysis  {best_name}', fontsize=13, fontweight='bold')

axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='crimson', linestyle='--', lw=1.5, label='Zero')
axes[0].axvline(residuals.mean(), color='orange', linestyle='--', lw=1.2,
                label=f'Mean = {residuals.mean():.3f}')
axes[0].set_title('Distribution\n(centred on 0 = no systematic bias)')
axes[0].set_xlabel('Residual  (Actual − Predicted)'); axes[0].legend(fontsize=8)

axes[1].scatter(best_pred, residuals, alpha=0.2, s=10, color='teal')
axes[1].axhline(0, color='crimson', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Rating'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted\n(random scatter = good)')

axes[2].scatter(y_test, residuals, alpha=0.2, s=10, color='purple')
axes[2].axhline(0,              color='crimson', linestyle='--', lw=1.5)
axes[2].axhline(+best_m['MAE'], color='orange',  linestyle=':', lw=1.2,
                label=f'+MAE ({best_m["MAE"]})')
axes[2].axhline(-best_m['MAE'], color='orange',  linestyle=':', lw=1.2, label='-MAE')
axes[2].set_xlabel('Actual Rating'); axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Actual\n(model struggles at extremes?)'); axes[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig('plot_09_residuals.png')
plt.close()
print("[Saved] plot_09_residuals.png")

# ── Plot 10: Learning Curve ──────────────────────────────────
print("\nComputing learning curve...")
train_sizes, tr_sc, val_sc = learning_curve(
    best_model, X_train, y_train, cv=3,
    scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=2)

tr_rmse  = -tr_sc
val_rmse = -val_sc

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, tr_rmse.mean(axis=1),  'o-', color='steelblue', label='Training RMSE')
ax.fill_between(train_sizes,
    tr_rmse.mean(axis=1) - tr_rmse.std(axis=1),
    tr_rmse.mean(axis=1) + tr_rmse.std(axis=1), alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_rmse.mean(axis=1), 'o-', color='crimson',   label='Validation RMSE')
ax.fill_between(train_sizes,
    val_rmse.mean(axis=1) - val_rmse.std(axis=1),
    val_rmse.mean(axis=1) + val_rmse.std(axis=1), alpha=0.15, color='crimson')
ax.axhline(BASELINE_RMSE, color='gray', linestyle='--', lw=1.2,
           label=f'Naive baseline ({BASELINE_RMSE})')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('RMSE')
ax.set_title(f'Learning Curve  {best_name}', fontweight='bold'); ax.legend()
plt.tight_layout()
plt.savefig('plot_10_learning_curve.png')
plt.close()
print("[Saved] plot_10_learning_curve.png")



[Saved] plot_06_model_comparison.png
[Saved] plot_07_tuning_results.png
[Saved] plot_08_actual_vs_predicted.png
[Saved] plot_09_residuals.png

Computing learning curve...
[Saved] plot_10_learning_curve.png


In [10]:
# Saving best model
joblib.dump(best_model, 'bookly_model.pkl')
print(f"\nSaved: bookly_model.pkl")

print(f"""
═══════════════════════════════════════════════════════════
                    TRAINING COMPLETE
═══════════════════════════════════════════════════════════
  Winner          : {best_name}
  Test RMSE       : {best_m['RMSE']}   (baseline: {BASELINE_RMSE})
  Test MAE        : {best_m['MAE']}
  Test R²         : {best_m['R²']}

  Saved artifacts :
    bookly_model.pkl        ← load with joblib.load()
    publisher_encoding.pkl  ← from Step 3
    author_encoding.pkl     ← from Step 3
    global_mean.pkl         ← from Step 3
    feature_columns.json    ← from Step 3
═══════════════════════════════════════════════════════════
""")


Saved: bookly_model.pkl

═══════════════════════════════════════════════════════════
                    TRAINING COMPLETE
═══════════════════════════════════════════════════════════
  Winner          : Random Forest (tuned)
  Test RMSE       : 0.2573   (baseline: 0.2996)
  Test MAE        : 0.184
  Test R²         : 0.2596

  Saved artifacts :
    bookly_model.pkl        ← load with joblib.load()
    publisher_encoding.pkl  ← from Step 3
    author_encoding.pkl     ← from Step 3
    global_mean.pkl         ← from Step 3
    feature_columns.json    ← from Step 3
═══════════════════════════════════════════════════════════

